# Faz 2 — Baseline Model

**Adım 1:** Label Encoding + Train/Test Split (%80/%20)  
**Adım 2:** Pipeline — VarianceThreshold → StandardScaler  
**Adım 3:** Random Forest — 5-Fold Stratified CV

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import joblib
import json
import time
from pathlib import Path

OUTPUTS = Path("../outputs")
RESULTS_DIR = OUTPUTS / "results"
MODELS_DIR  = OUTPUTS / "models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Veriyi Yükle

In [ ]:
df = pd.read_parquet(OUTPUTS / "cicids2017_clean.parquet")
print(f"Satır: {len(df):,}  |  Sütun: {df.shape[1]}")
print(f"Sınıflar: {df['Label'].nunique()}")
df['Label'].value_counts()

## 2. Label Encoding

In [ ]:
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['Label'])

# Sınıf → integer eşlemesini kaydet
label_map = {cls: int(idx) for idx, cls in enumerate(le.classes_)}
with open(RESULTS_DIR / "label_map.json", "w") as f:
    json.dump(label_map, f, indent=2, ensure_ascii=False)

# Encoder'ı diske kaydet (Faz 3+ için tekrar kullanılacak)
joblib.dump(le, MODELS_DIR / "label_encoder.joblib")

print("Label map:")
for cls, idx in label_map.items():
    print(f"  {idx:2d} → {cls}")

## 3. Train / Test Split  (%80 / %20 — Stratified)

In [ ]:
feature_cols = [c for c in df.columns if c not in ("Label", "label_enc")]
X = df[feature_cols]
y = df["label_enc"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train: {len(X_train):,} satır  ({len(X_train)/len(df)*100:.1f}%)")
print(f"Test : {len(X_test):,} satır  ({len(X_test)/len(df)*100:.1f}%)")
print(f"\nFeature sayısı: {X_train.shape[1]}")

In [ ]:
# Stratifikasyonu doğrula: her sınıfın oranı korunmuş mu?
train_dist = y_train.value_counts(normalize=True).rename("train_%")
test_dist  = y_test.value_counts(normalize=True).rename("test_%")
dist_check = pd.concat([train_dist, test_dist], axis=1).sort_index()
dist_check.index = le.inverse_transform(dist_check.index)
dist_check = dist_check * 100
dist_check["fark"] = (dist_check["train_%"] - dist_check["test_%"]).abs()
print(dist_check.round(4).to_string())
print(f"\nMaks. sapma: {dist_check['fark'].max():.5f}%  (0'a yakın olmalı ✓)")

## 4. Pipeline: VarianceThreshold → StandardScaler

In [ ]:
# VarianceThreshold eşiği: EDA'da 4 adet çok düşük varyanslı sütun tespit edildi.
# threshold=0.01 → bu sütunlar çıkarılır, geri kalan her şey StandardScaler'a girer.
preprocessor = Pipeline([
    ("var_thresh", VarianceThreshold(threshold=0.01)),
    ("scaler",     StandardScaler()),
])

preprocessor.fit(X_train)

# Kaç feature kaldı?
mask        = preprocessor.named_steps["var_thresh"].get_support()
kept_cols   = X_train.columns[mask].tolist()
dropped_cols = X_train.columns[~mask].tolist()

print(f"Başlangıç feature sayısı : {X_train.shape[1]}")
print(f"VarianceThreshold sonrası: {len(kept_cols)}  (düşürülen: {len(dropped_cols)})")
if dropped_cols:
    print(f"  Çıkarılan sütunlar: {dropped_cols}")

In [ ]:
# Preprocessor'ı dönüştür → sonraki adımlarda (CV, SMOTE) hazır matris kullanılacak
X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f"X_train_proc shape: {X_train_proc.shape}")
print(f"X_test_proc  shape: {X_test_proc.shape}")
print(f"\nScaler mean  (ilk 5): {preprocessor.named_steps['scaler'].mean_[:5].round(4)}")
print(f"Scaler scale (ilk 5): {preprocessor.named_steps['scaler'].scale_[:5].round(4)}")

In [ ]:
# Preprocessor'ı kaydet (Faz 3+ aynı pipeline'ı kullanacak)
joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

# Kalan feature isimlerini kaydet
with open(RESULTS_DIR / "kept_features.json", "w") as f:
    json.dump(kept_cols, f, indent=2)

print("preprocessor.joblib  →", MODELS_DIR / "preprocessor.joblib")
print("kept_features.json   →", RESULTS_DIR / "kept_features.json")

## 5. Random Forest — 5-Fold Stratified CV

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# %10 örnekleme — stratified, tekrarlanabilir
sample_frac = 0.1
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(y_train), size=int(len(y_train) * sample_frac), replace=False)
idx.sort()

X_cv = X_train_proc[idx]
y_cv = y_train.iloc[idx].reset_index(drop=True)

print(f"CV için örnekleme: {len(y_cv):,} satır  (toplam train'in %{sample_frac*100:.0f}'i)")
print(f"CV başlıyor... ({len(y_cv):,} satır × 67 feature × 5 fold)")
t0 = time.time()

cv_results = cross_validate(
    rf, X_cv, y_cv,
    cv=cv,
    scoring=["accuracy", "f1_macro", "roc_auc_ovr_weighted"],
    return_train_score=True,
    n_jobs=1,
    verbose=1,
)

elapsed = time.time() - t0
print(f"\nToplam süre: {elapsed/60:.1f} dakika")

In [ ]:
# Fold sonuçlarını DataFrame'e aktar
folds_df = pd.DataFrame({
    "fold":               range(1, 6),
    "train_accuracy":     cv_results["train_accuracy"],
    "val_accuracy":       cv_results["test_accuracy"],
    "train_f1_macro":     cv_results["train_f1_macro"],
    "val_f1_macro":       cv_results["test_f1_macro"],
    "train_roc_auc":      cv_results["train_roc_auc_ovr_weighted"],
    "val_roc_auc":        cv_results["test_roc_auc_ovr_weighted"],
    "fit_time_s":         cv_results["fit_time"],
    "score_time_s":       cv_results["score_time"],
})

folds_df.to_csv(RESULTS_DIR / "random_forest_cv_folds.csv", index=False)
print(folds_df.to_string(index=False))

print("\n--- Ortalama ± Std ---")
for metric in ["val_accuracy", "val_f1_macro", "val_roc_auc"]:
    vals = folds_df[metric]
    print(f"  {metric:20s}: {vals.mean():.4f} ± {vals.std():.4f}")